In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import qutip as qp

from tqdm import tqdm

In [ ]:
from tensor_networks_simulations.mps.states import MPDO, MPS
from tensor_networks_simulations.mps.algorithms import tebd_alg_choi, one_time_step_2nd_order, one_time_step_2nd_order_2approach, tebd_2nd_order_LPTN_thermal
from tensor_networks_simulations.mps.models import BondHamiltonian, H_bond_choi, HBondChoi
from tensor_networks_simulations.mps.tools import (
    correlation_one_site_mix, 
    projection_Normalization_Mix, 
    T_spin_correl_mix, 
    apply_one_site_op_mix_state, 
    expectation_value_op, 
    expectation_value, 
    apply_one_site_op_mpo
)
from tensor_networks_simulations.general_tools import ED_tools as sf
from tensor_networks_simulations.general_tools.plot_tools import figure_styling
figure_styling()

In [ ]:
L = 8
hzs = .6 * np.ones(L) 
Jzs = -0.0 * np.ones(L)
Jxs = [1.]*L
Jys = [1.]*L
hxs = mus = -0.0 * np.ones(L)
gamma= 0.0
chi_max = 32
dt=0.005
dt_list = np.array([1j*dt,]*L)
scale = 1
tot_steps = int(4/(scale*dt))


Hb = BondHamiltonian(L, Jxs, Jys, Jzs, hxs, hzs, mus)
Hs = [Hb.nn_term(i) for i in range(L-1)]

In [ ]:
def mpo_inf_temperature(L):
    Bs_inf = MPDO().infinite_temp_state(L)
    bonds_inf = MPDO().bond_vec_inf_temp(L)
    Bs_inf, Ss_inf = MPDO().schmidt_vals_from_mps(Bs_inf, bonds_inf)
    Ms_inf, Ss_inf = MPDO().mpo_from_purified_mps(Bs_inf, Ss_inf)
    bonds_inf = bonds_inf[::2]
    bonds_inf.append(1)
    mpo_inf = MPDO(Ms_inf, Ss_inf, bonds_inf)
    return mpo_inf 


mpo_inf = mpo_inf_temperature(L)
#mpo_inf = MPDO.update_mpo_with_schmidt_vals(mpo_inf)


phi_inf = copy.deepcopy(mpo_inf)
phi_inf = apply_one_site_op_mpo(phi_inf, Hb.sz, int(L/2))



   
T_spin_correl_mix(phi_inf.Ms, Hb.s0, phi_inf.Ms, L, int(L/2)), projection_Normalization_Mix(mpo_inf.Ms, mpo_inf.Ms, L)


In [ ]:
# rho = MPDO.from_mpo(mpo_inf)

# projection_Normalization_Mix(rho.Ms, rho.Ms, L)
# rho = MPDO.update_mpo_with_schmidt_vals(rho)
# projection_Normalization_Mix(rho.Ms, rho.Ms, L)

In [ ]:

mag_t = []
SzSz_ts =[]
mx_ts_norms = []
ts = []
t=0
deltat=0
disc_err = []
discerr = 0.
for i in tqdm(range(tot_steps)):
    mpo_inf, disc1 =  tebd_2nd_order_LPTN(mpo_inf, Hs, chi_max, L, dt_list, epsilon=10 ** (-5))
    phi_inf, disc2 =  tebd_2nd_order_LPTN(phi_inf, Hs, chi_max, L, dt_list, epsilon=10 ** (-5))
    
    t += dt
    discerr += disc1 + disc2
    if np.mod(i,20)==0:
        #rho = MPDO.from_mpo(mpo)
        SzSz = T_spin_correl_mix(mpo_inf.Ms, Hb.sz, phi_inf.Ms, L, int(L/2))
        Mz = np.sum([T_spin_correl_mix(phi_inf.Ms, Hb.sz, phi_inf.Ms, L, i) for i in range(L)])/L
        
        # print(f"""
        #       norm = {T_spin_correl_mix(mpo_inf.Ms, Hb.s0, mpo_inf.Ms, L, 0):.3f}, 
        #       <Sz(t)Sz> = {SzSz:.3f}, Mz={Mz.real:.3f},
        #       t={t:.2f}, disc = {discerr:.4f},
        #       max_bond={max(mpo_inf.bonds)},{max(phi_inf.bonds)}
        #       """)
        
        mag_t.append(Mz)
        SzSz_ts.append(SzSz)
        
        disc_err.append(discerr)
        ts.append(t)

    
   
    


In [ ]:
plt.plot(ts, np.array(SzSz_ts)/4, marker = 'o',ms=1.5, label='$<S_x> TEBD$')
plt.ylim(-0.04,0.1)

In [ ]:
solver = "me"   # use the ode solver
#solver = "mc"   # use the monte-carlo solver


# decoherence rate
gammas = gamma * np.ones(L)
print(gammas)

init_ghz = [(qp.basis(2, 0)+qp.basis(2, 1))/np.sqrt(2.)]*L
init = [qp.basis(2, 0), qp.basis(2, 0)]*int(L/2)
#print(init)




# intial state, first spin in state |1>, the rest in state |0>
psi_list = []
psi_list.append(qp.basis(2,1))
for n in range(L-1):
    psi_list.append(qp.basis(2,0))
    
    
psi0 = qp.tensor(init_ghz) 
#psi0 = qp.ghz_state(L)

tlist = np.linspace(0, 5, 100)

expts = sf.integrate(L, hzs, Jxs, Jys, Jzs, psi0, tlist, gammas, solver)

In [ ]:
mz_ex = []
mx_ex = []
for i, t in enumerate(tlist):
    mz_ex.append(np.sum([expts[j][i] for j in range(L)])/L)
    mx_ex.append(np.sum([expts[j][i] for j in range(L, 2*L)])/L)

In [ ]:
import matplotlib.pyplot as plt
csfont = {'fontname':'Comic Sans MS'}
hfont = {'fontname':'Helvetica'}

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16,6))
ax[0].plot(tlist, np.array(mz_ex), label="exact $<S_z>$")
ax[0].plot(tlist, 1*np.array(mx_ex),label="exact $<S_x>$")


ax[1].plot(ts, disc_err)
ax[1].set_yscale('log')
ax[0].plot(ts, mag_t, marker="s", ms=1, label='$<S_z> TEBD$')
ax[0].plot(ts, 1*np.array(mx_ts),marker="s", ms=3, ls = ":", label='$<S_x> TEBD$')
ax[0].plot(ts, 1*np.array(mx_ts_norms),marker="s", ms=2.5, ls = ":", label='$<S_x> TEBD~normalized$')

ax[0].set_xlabel('Time')
ax[1].set_xlabel('Time')
ax[1].set_ylabel('Truncation error')
ax[0].set_ylabel('Expectation values') 
ax[0].legend(fontsize=16)
plt.legend(fontsize=16)  
plt.tight_layout()
plt.show() 

In [ ]:
psi0

In [ ]:
import _pickle as cPickle

In [ ]:
#import pickle as cPickle
mylist = ['apple', 'bat', 'cat', 'dog']
with open('data.pickle', 'wb') as fh:
    cPickle.dump(mylist, fh)

In [ ]:
cPickle_off = open("data_P_A.pickle", "rb")
szsz_pa = cPickle.load(cPickle_off)
print(szsz_pa)

In [ ]:
cPickle_off = open("data_P.pickle", "rb")
szsz_p = cPickle.load(cPickle_off)
print(szsz_p)

In [ ]:
tot_steps = int(16/(0.01))
tlist = range(0, tot_steps, 20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(16,6))
ax.plot(tlist, np.array(szsz_pa),'o' ,label="P_A")
ax.plot(tlist, 1*np.array(szsz_p),'s',label="P")




ax.set_xlabel('Time')
ax.set_xlabel('Time')
ax.set_ylabel('Truncation error')
ax.set_ylabel('Expectation values') 
ax.legend(fontsize=16)
plt.legend(fontsize=16)  
plt.tight_layout()
plt.show() 

In [ ]:
def kraus_op_svd(op, dt=0.01, d=2):
    idn = np.reshape(np.eye(d), (d*d, 1))
    op1 = np.reshape(op, (d*d, 1))
    op1 = np.kron(op1, op1.T)
    
    op2 = np.conj(op.T) @ op
    op2 = np.reshape(op2, (d*d, 1))
    op2 = np.kron(op2, idn.T)
    
    op3 = op.T @ np.conj(op)
    op3 = np.reshape(op3, (1, d*d))
    op3 = np.kron(idn, op3)
    
    D_op = op1 -0.*op2 - 0.*op3
    
    
    eop = expm(dt*D_op)
    # print(f"D =\n {eop}, \n---\n {eop.T}")
    u,s,v = svd(eop)
    #s = s/sum(s**2)
    s = s**(1/2)
    S = np.diag(s)#**1/2
    #S = S/np.sum(S**2)
    B = u.dot(S)
    B_dag = S.dot(v)
    
    eop1 = B @ B.T
    assert eop1.all() == eop.all()
    B = np.reshape(B, (2,2,4))
    return B

In [ ]:
#mpo_inf = mpo_inf_temperature(L)

In [ ]:
from scipy.linalg import expm, svd
def apply_noise_evolution(mpo: MPDO, op, dt, d=2, Kr = 3):
    B = kraus_op_svd(op, dt, d)
    M_list = []
    for i in range(len(mpo.Ms)):
        M = np.tensordot(B, mpo.Ms[i], axes=([1],[0])) #  (p, k, q, a_l, a_r)
        M = np.transpose(M, (1,2,0,3,4))   #  (k, q, p, a_l, a_r)
        k = M.shape[0]; q=M.shape[1]; a_l = M.shape[3]; a_r = M.shape[4]
        M = np.reshape(M, (k*q, d*a_l*a_r))  #  (k*q, p*a_l*a_r)
        X, Y, Z = svd(M)   # (k*q, gamma), (gamma), (gamma, p*a_l*a_r)
        tmp = np.linalg.norm(Y[: Kr])
        S = Y[: Kr] / tmp
        Z = Z[:Kr, :]
        M_new = np.diag(S) @ Z
        print(M_new.shape)
        M_new = np.reshape(M_new, (Kr, d, a_l, a_r))
        print(M_new.shape)
        M_new = np.transpose(M_new, (1, 0, 2,3))
        M_list.append(M_new)
    mpo.Ms = M_list
    return mpo

In [ ]:
mpo = apply_noise_evolution(mpo_inf, Hb.sx, dt=0.01)

In [ ]:
mpo.Ms[1].shape

In [ ]:
mag_t = []
SzSz_ts =[]
mx_ts_norms = []
ts = []
t=0
deltat=0
disc_err = []
discerr = 0.
for i in tqdm(range(tot_steps)):
    mpo_inf, disc1 =  tebd_2nd_order_LPTN(mpo_inf, Hs, chi_max, L, dt_list, epsilon=10 ** (-5))
    phi_inf, disc2 =  tebd_2nd_order_LPTN(phi_inf, Hs, chi_max, L, dt_list, epsilon=10 ** (-5))
    
    t += dt
    discerr += disc1 + disc2
    if np.mod(i,20)==0:
        #rho = MPDO.from_mpo(mpo)
        SzSz = T_spin_correl_mix(mpo_inf.Ms, Hb.sz, phi_inf.Ms, L, int(L/2))
        Mz = np.sum([T_spin_correl_mix(phi_inf.Ms, Hb.sz, phi_inf.Ms, L, i) for i in range(L)])/L
        
        # print(f"""
        #       norm = {T_spin_correl_mix(mpo_inf.Ms, Hb.s0, mpo_inf.Ms, L, 0):.3f}, 
        #       <Sz(t)Sz> = {SzSz:.3f}, Mz={Mz.real:.3f},
        #       t={t:.2f}, disc = {discerr:.4f},
        #       max_bond={max(mpo_inf.bonds)},{max(phi_inf.bonds)}
        #       """)
        
        mag_t.append(Mz)
        SzSz_ts.append(SzSz)
        
        disc_err.append(discerr)
        ts.append(t)
